# 05c. 품질 점수표 · 민감도 · 정답지 오염 (판정기준서 v1.1 9장, D64, 2026-11-23)

- 점수표: 기준 ①~④·이관·원천·G별 대상 수, 결함 수·비율, 처리, Training/Validation 따로, 사람 확인 수, 근거.
- 민감도: 공개 기준을 성향마다 한 등급 느슨하게(적극 1–6, 위험중립 3–6, 안정추구 4–6, 안정 5–6) 했을 때 ③-1(공개 기준 밖 추천)에 남는 추천·CoT 수.
- 정답지 오염: Validation의 기준별 결함 비율을 Training과 나란히 (P4).
- 숫자만 둔다. 해석은 쓰지 않는다.

데이터: AI Hub(한국지능정보사회진흥원) 사업 결과

In [ ]:
import sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
pd.set_option("display.width", 220)
from src.load_cot import load_cot, load_products
from src.judge import judge
from src.report import update_scorecard
AS_OF = "2026-11-24"
cot = load_cot(); prod = load_products()
j, ctx = judge(cot, prod, as_of=AS_OF)
g4 = pd.read_csv(ROOT / "outputs" / "latest_grade_rejudge.csv", encoding="utf-8-sig", keep_default_na=False)
lg = pd.read_csv(ROOT / "outputs" / "latest_grade_check.csv", encoding="utf-8-sig", keep_default_na=False)
SP = ["Training", "Validation"]
N = j["분할"].value_counts()

## 1. 품질 점수표 (CoT 단위)

In [ ]:
def item(crit, name, mask, base, action, basis, human=0):
    r = {"기준": crit, "지표": name, "처리": action, "근거": basis, "사람 확인": human}
    for s in SP + ["전체"]:
        b = base if s == "전체" else base & (j["분할"] == s)
        m = mask & b
        r[f"대상_{s}"], r[f"결함_{s}"] = int(b.sum()), int(m.sum())
        r[f"비율_{s}"] = round(m.sum() / b.sum() * 100, 1) if b.sum() else float("nan")
    return r
ALL = pd.Series(True, index=j.index)
rows = [
    item("①", "법정 3항목 중 미확인 있음", j["기준① 법정 미확인"] != "", ALL, "점수 보고", "금소법 제17조②2호"),
    item("①", "6항목 중 확인 4개 이하", j["기준① 확인 수"].astype(int) <= 4, ALL, "점수 보고", "판정기준서 1장"),
    item("②", "②-a 근거 부족 확정", j["기준② 판정"] == "②-a 근거 부족 확정", ALL, "태그", "금소법 제17조②2호·③, D36"),
    item("②", "②-b 성향 미확정 추천", j["기준② 판정"] == "②-b 성향 미확정 추천", ALL, "태그", "금소법 제17조②·③"),
    item("③", "범위 밖 (제외)", j["기준③ 처리"] == "제외", ALL, "제외", "KIS 준칙 제6조①, D55·D57·D52·D47",
         human=int((j["기준③ 판정"] == "판정 불가(범위 표기 손상)").sum())),
    item("③", "판정 불가 (성향·추천·등급 없음, 표기 손상)", j["기준③ 판정"].str.startswith("판정 불가"), ALL, "판정 불가", "D21·D34·D10·D57"),
    item("③태그", "공개 기준보다 넓은 허용 범위", j["D30 태그"] != "", ALL, "태그(교정판 교정)", "D30"),
    item("④", "일부 기재", j["기준④ 판정"] == "일부 기재", ALL, "태그 / 교정판 등급 문장", "금소법 제19조①, D19"),
    item("④", "기재 없음", j["기준④ 판정"] == "기재 없음", ALL, "태그 / 교정판 등급 문장", "금소법 제19조①, D19"),
    item("이관", "성향 밖 등급 직접 추천·이관 없음", j["이관 판정"] == "결함", ALL, "태그", "KIS 준칙 제6조③, 표준준칙 10.4)·회사참고사항 10-4",
         human=int((j["이관 판정"] == "사람 확인").sum())),
    item("원천", "원천 확인 필요 상품 추천", j["원천 확인 필요"].astype(str) == "True", ALL, "표시", "D51·D59"),
]
gt = set(g4.loc[g4["G4 태그"] != "", "cot_id"].astype(int))
gb = set(g4["cot_id"].astype(int))
rows.append(item("G", "최신 기준 부적합 (상위 30개 상품 추천 CoT)", j["cot_id"].isin(gt), j["cot_id"].isin(gb), "태그", "G4",
                 human=int((lg["등급 변경"] == "변경").sum())))
sc = pd.DataFrame(rows)
print(sc[["기준", "지표", "결함_Training", "비율_Training", "결함_Validation", "비율_Validation", "결함_전체", "비율_전체", "처리", "사람 확인"]].to_string(index=False))

## 2. 사람 확인 사례 (판정기준서 7장)

In [ ]:
human = {
    "③-3 범위 표기 손상 (CoT)": int((j["기준③ 판정"] == "판정 불가(범위 표기 손상)").sum()),
    "이관 규칙으로 분류할 수 없는 문장 (CoT)": int((j["이관 판정"] == "사람 확인").sum()),
    "G4 등급이 바뀐 상품 (상품)": int((lg["등급 변경"] == "변경").sum()),
    "G 조회 명칭 다름 (상품)": int(lg["상태"].str.startswith("사람 확인").sum()),
    "원천 확인 필요 상품 중 공식 문서 미확인 (상품)": len(ctx["mismatch_keys"]) - 1,
    "C안 LLM·규칙 불일치 (B안이라 해당 없음)": 0,
}
for k, v in human.items(): print(f"{k}: {v}")
print("※ 원천 확인 필요 상품 수는 body_grade와 등록 등급이 다른 상품 수에서 공식 문서로 확인한 NH-Amundi 베트남 레버리지 1개를 뺀 값")

## 3. 민감도 — 공개 기준을 한 등급 느슨하게

In [ ]:
RELAX = {"공격투자형": set(range(1, 7)), "적극투자형": set(range(1, 7)), "위험중립형": set(range(3, 7)),
         "안정추구형": set(range(4, 7)), "안정형": {5, 6}}
recs = ctx["recs"].copy()
recs["성향"] = recs["cot_id"].map(ctx["cots"]["성향_D21"])
recs["분할"] = recs["cot_id"].map(ctx["cots"]["split"])
ok = recs["json_grade"].notna() & recs["성향"].isin(RELAX)
def outside(r):
    return [(int(g) not in r[p]) if o else False for g, p, o in zip(recs["json_grade"], recs["성향"], ok)]
recs["공개 기준 밖"] = outside(ctx["range"]); recs["완화 기준 밖"] = outside(RELAX)
sens =pd.concat({s: pd.DataFrame({k: {"추천": int((recs[k] & (recs["분할"] == s if s != "전체" else True)).sum()),
                                       "CoT": recs.loc[recs[k] & (recs["분할"] == s if s != "전체" else True), "cot_id"].nunique()}
                                   for k in ["공개 기준 밖", "완화 기준 밖"]}) for s in SP + ["전체"]}, axis=1)
print(sens)
print("완화 후 남는 비율 (CoT):", round(sens[("전체", "완화 기준 밖")]["CoT"] / sens[("전체", "공개 기준 밖")]["CoT"] * 100, 1), "%")

## 4. 정답지 오염 — Validation과 Training 결함 비율

In [ ]:
cont = sc[["기준", "지표", "비율_Training", "비율_Validation", "결함_Validation", "대상_Validation"]].copy()
cont["차이(V-T)"] = (cont["비율_Validation"] - cont["비율_Training"]).round(1)
print(cont.to_string(index=False))

## 5. scorecard 저장

In [ ]:
def srow(name, t, v, a, unit, src, basis):
    return {"지표": name, "Training": t, "Validation": v, "전체": a, "단위": unit, "출처": src, "근거 결정": basis, "기준일": AS_OF}
out = []
for _, r in sc.iterrows():
    out.append(srow(f"{r['기준']} {r['지표']} (결함 수)", r["결함_Training"], r["결함_Validation"], r["결함_전체"], "CoT", "05c_품질점수표", r["근거"]))
    out.append(srow(f"{r['기준']} {r['지표']} (비율 %)", r["비율_Training"], r["비율_Validation"], r["비율_전체"], "%", "05c_품질점수표", r["근거"]))
for k, v in human.items():
    out.append(srow(f"사람 확인: {k}", "", "", v, "건", "05c_품질점수표", "판정기준서 7장"))
for k in ["공개 기준 밖", "완화 기준 밖"]:
    for u in ["추천", "CoT"]:
        out.append(srow(f"민감도: {k} ({u})", sens.loc[u, ("Training", k)], sens.loc[u, ("Validation", k)], sens.loc[u, ("전체", k)], u, "05c_품질점수표", "판정기준서 9장"))
# ③ 공개 기준 밖 추천 건수 (D54의 대표 수치 799를 최신 판정으로 갱신, 등록 등급·성향_D21(P5 보충·우선 반영))
nrec = {s: int((recs["분할"] == s).sum()) for s in SP}; nrec["전체"] = len(recs)
oc = {s: sens.loc["추천", (s, "공개 기준 밖")] for s in SP + ["전체"]}
out.append(srow("③ 공개 기준 밖 추천 (추천 수, D54 대표 수치 갱신, 이전 799)", oc["Training"], oc["Validation"], oc["전체"], "추천", "05c_품질점수표", "D43·D54·D65"))
out.append(srow("③ 공개 기준 밖 추천 / 증권 추천 전체 (%)", *[round(oc[s] / nrec[s] * 100, 1) for s in SP + ["전체"]], "%", "05c_품질점수표", "D43·D54·D65"))
out.append(srow("증권 추천 전체", nrec["Training"], nrec["Validation"], nrec["전체"], "추천", "05c_품질점수표", ""))
out.append(srow("CoT 수", N["Training"], N["Validation"], len(j), "CoT", "05c_품질점수표", ""))
update_scorecard(out, "품질 점수표 (판정기준서 v1.3)")
print("저장:", len(out), "행")

---
데이터: AI Hub(한국지능정보사회진흥원) 사업 결과